# CSE 654 — ModernBERT-TR ve XLM-R (BERTurk ile aynı kurulum)
- **Aynı bölme:** `splits/split_ids.csv` okunur, yeniden oluşturulmaz. Test çiftleri BERTurk'ün test çiftleriyle birebir aynı mı diye kontrol edilir.
- **Aynı ayarlar:** LR 2e-5, batch 32, 4 epoch, warmup %10, weight decay 0.01, seed 42, fp16, en iyi epoch dev macro-F1 ile seçilir.
- **Tek fark:** model.
- Notebook'u baştan sona çalıştır. Biten model tekrar eğitilmez; oturum kapanırsa eğitim son checkpoint'ten devam eder.

Çalışma zamanı: **T4 GPU**

## Drive bağlantısı

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJE     = '/content/drive/MyDrive/cse654_absa'
VERI      = f'{PROJE}/absa_10k_final.csv'
SPLIT_DIR = f'{PROJE}/splits'
SONUC_DIR = f'{PROJE}/sonuclar'
SPLIT_FILE = f'{SPLIT_DIR}/split_ids.csv'
for p in [VERI, SPLIT_FILE, f'{SONUC_DIR}/berturk_test_tahminleri.csv']:
    assert os.path.exists(p), f'Bulunamadı: {p}'
print('Hazır:', PROJE)

Mounted at /content/drive
Hazır: /content/drive/MyDrive/cse654_absa


## Kütüphaneler

In [2]:
!pip -q install transformers accelerate datasets

## Ayarlar

In [3]:
import ast, re, math, inspect, gc
import numpy as np
import pandas as pd
import torch, transformers
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (AutoConfig, AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from transformers.trainer_utils import get_last_checkpoint
from datasets import Dataset

print('transformers', transformers.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK, T4 GPU seç!')

SEED = 42
# BERTurk ile birebir aynı eğitim ayarları
LR, BATCH, EPOCHS, WARMUP, WD = 2e-5, 32, 4, 0.1, 0.01

MODELLER = {
    'berturk':       'dbmdz/bert-base-turkish-cased',      # zaten eğitildi, sadece sonuçları okunur
    'modernbert_tr': 'ytu-ce-cosmos/modernbert-tr-base',
    'xlmr':          'FacebookAI/xlm-roberta-base',
}
CALISTIR = ['modernbert_tr', 'xlmr']   # bu notebook'ta eğitilecekler

LABELS   = ['negative', 'neutral', 'positive']
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

transformers 5.22.0 | GPU: Tesla T4


## Veri ve aynı bölme

In [4]:
df = pd.read_csv(VERI, dtype=str, keep_default_na=False)
df['id'] = df['id'].astype(int)
df['targets']    = df['target'].map(ast.literal_eval)
df['polarities'] = df['polarity'].map(ast.literal_eval)
assert all(len(t) == len(p) for t, p in zip(df.targets, df.polarities))

def grup(l):
    if len(l) == 1:                         return 'tek_hedefli'
    if len(set(l)) == 1:                    return 'ayni_duygulu'
    if {'positive', 'negative'} <= set(l):  return 'zit_duygulu'
    return 'notrle_karisik'
df['grup']      = df.polarities.map(grup)
df['n_targets'] = df.targets.map(len)
df['kelime']    = df.text.str.split().map(len)
df['baglac']    = df.text.str.lower().map(lambda s: int(bool(re.search(r'\b(?:ama|fakat|ancak|lakin|ne var ki|oysa)\b', s))))

# BÖLME: BERTurk'te kullanılan kayıtlı bölme okunur, yeniden oluşturulmaz
df = df.merge(pd.read_csv(SPLIT_FILE), on='id', how='left')
assert df['split'].notna().all(), 'split_ids.csv bütün id\'leri içermiyor.'
display(pd.crosstab(df.grup, df.split, margins=True)[['train', 'dev', 'test', 'All']])

rows = []
for r in df.itertuples():
    for k, (t, p) in enumerate(zip(r.targets, r.polarities), start=1):
        rows.append(dict(id=r.id, sira=k, text=r.text, target=t, label=p, category=r.category,
                         grup=r.grup, n_targets=r.n_targets, kelime=r.kelime, baglac=r.baglac, split=r.split))
pairs = pd.DataFrame(rows)
train_df = pairs[pairs.split == 'train'].reset_index(drop=True)
dev_df   = pairs[pairs.split == 'dev'].reset_index(drop=True)
test_df  = pairs[pairs.split == 'test'].reset_index(drop=True)

# KONTROL: test çiftleri BERTurk'ün test çiftleriyle birebir aynı mı?
bt = pd.read_csv(f'{SONUC_DIR}/berturk_test_tahminleri.csv')
ayni_test = (set(zip(bt.id, bt.sira)) == set(zip(test_df.id, test_df.sira))) and (len(bt) == len(test_df))
assert ayni_test, 'Test seti BERTurk ile aynı değil! split_ids.csv değişmiş olabilir.'
print(f'Train {len(train_df):,} | dev {len(dev_df):,} | test {len(test_df):,} çift — test seti BERTurk ile birebir aynı ✔')

split,train,dev,test,All
grup,,,,
ayni_duygulu,3610,401,1003,5014
notrle_karisik,1607,179,447,2233
tek_hedefli,570,63,159,792
zit_duygulu,1411,157,392,1960
All,7198,800,2001,9999


Train 19,715 | dev 2,168 | test 5,407 çift — test seti BERTurk ile birebir aynı ✔


## Eğitim fonksiyonları

In [5]:
def hazirla(tok, dfp, max_len, tt):
    ds = Dataset.from_pandas(dfp[['text', 'target', 'label']])
    def enc(b):
        out = tok(b['text'], b['target'], truncation='only_first', max_length=max_len, return_token_type_ids=tt)
        out['labels'] = [label2id[x] for x in b['label']]
        return out
    return ds.map(enc, batched=True, remove_columns=['text', 'target', 'label'])

def compute_metrics(p):
    y_hat, y = p.predictions.argmax(-1), p.label_ids
    return {'accuracy': accuracy_score(y, y_hat), 'macro_f1': f1_score(y, y_hat, average='macro')}

def egit(model_adi, tok, train_ds, dev_ds, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    transformers.set_seed(SEED)
    cfg = AutoConfig.from_pretrained(model_adi, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    if getattr(cfg, 'reference_compile', None) is not None or cfg.model_type == 'modernbert':
        cfg.reference_compile = False      # eski transformers sürümlerinde ModernBERT'in torch.compile'ını kapatır
    model = AutoModelForSequenceClassification.from_pretrained(model_adi, config=cfg)
    kw = dict(output_dir=out_dir, save_strategy='epoch', save_total_limit=2,
              load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True,
              learning_rate=LR, per_device_train_batch_size=BATCH, per_device_eval_batch_size=64,
              num_train_epochs=EPOCHS, warmup_steps=int(WARMUP * math.ceil(len(train_ds) / BATCH) * EPOCHS),
              weight_decay=WD, fp16=torch.cuda.is_available(), logging_steps=50, report_to='none', seed=SEED)
    kw['eval_strategy' if 'eval_strategy' in inspect.signature(TrainingArguments.__init__).parameters
       else 'evaluation_strategy'] = 'epoch'
    trainer = Trainer(model=model, args=TrainingArguments(**kw), train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    last = get_last_checkpoint(out_dir)
    print('Checkpoint bulundu, devam:' if last else 'Sıfırdan eğitim.', last or '')
    trainer.train(resume_from_checkpoint=last)
    kayiplar = [x['loss'] for x in trainer.state.log_history if 'loss' in x]
    if kayiplar and not np.isfinite(kayiplar[-1]):
        print('⚠ Eğitim kaybı NaN oldu. Bu model için fp16 sorun çıkarmış olabilir.')
    print('En iyi checkpoint:', trainer.state.best_model_checkpoint)
    return trainer

## ModernBERT-TR ve XLM-R eğitimi (sırayla)

In [6]:
for kisa in CALISTIR:
    model_adi = MODELLER[kisa]
    tahmin_dosyasi = f'{SONUC_DIR}/{kisa}_test_tahminleri.csv'
    if os.path.exists(tahmin_dosyasi):
        print(f'[{kisa}] zaten eğitilmiş, atlanıyor ({tahmin_dosyasi})'); continue
    print(f'\n========== {kisa}: {model_adi} ==========')

    cfg = AutoConfig.from_pretrained(model_adi)
    tt = cfg.model_type not in ('modernbert',)                 # ModernBERT token_type_ids kullanmıyor

    tok = AutoTokenizer.from_pretrained(model_adi)
    lens = np.array([len(x) for x in tok(pairs.text.tolist(), pairs.target.tolist())['input_ids']])
    max_len = int(min(512, math.ceil(np.percentile(lens, 99.9) / 32) * 32))   # BERTurk'teki kuralla aynı
    print(f'Token: ort {lens.mean():.0f}, %99 {np.percentile(lens, 99):.0f}, max {lens.max()} → MAX_LEN={max_len}')

    tr_ds, dv_ds, te_ds = (hazirla(tok, d, max_len, tt) for d in (train_df, dev_df, test_df))
    trainer = egit(model_adi, tok, tr_ds, dv_ds, f'{PROJE}/{kisa}_checkpoints')
    trainer.save_model(f'{PROJE}/{kisa}_final'); tok.save_pretrained(f'{PROJE}/{kisa}_final')

    out = test_df.copy()
    out['pred']  = [id2label[i] for i in trainer.predict(te_ds).predictions.argmax(-1)]
    out['dogru'] = (out.pred == out.label).astype(int)
    out.to_csv(tahmin_dosyasi, index=False)
    print(classification_report(out.label, out.pred, labels=LABELS, digits=4))
    print('Kaydedildi:', tahmin_dosyasi)

    del trainer, tr_ds, dv_ds, te_ds; gc.collect(); torch.cuda.empty_cache()


========== modernbert_tr: ytu-ce-cosmos/modernbert-tr-base ==========


config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 50007), got 50282. This may result in unexpected behavior.
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 50007), got 50281. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

... (34 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

En iyi checkpoint: /content/drive/MyDrive/cse654_absa/xlmr_checkpoints/checkpoint-2468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

    negative     0.8878    0.8981    0.8929      1806
     neutral     0.7743    0.7369    0.7551      1220
    positive     0.8983    0.9126    0.9054      2381

    accuracy                         0.8681      5407
   macro avg     0.8535    0.8492    0.8512      5407
weighted avg     0.8668    0.8681    0.8673      5407

Kaydedildi: /content/drive/MyDrive/cse654_absa/sonuclar/xlmr_test_tahminleri.csv


## Karşılaştırma (3 model, aynı test seti)
BERTurk'ün sonuçları Drive'daki tahmin dosyasından okunur; bütün modeller aynı fonksiyonlarla değerlendirilir.

## 3 modelin karşılaştırması

In [7]:
def skor(d):
    return pd.Series({'accuracy': accuracy_score(d.label, d.pred),
                      'macro_f1': f1_score(d.label, d.pred, labels=LABELS, average='macro', zero_division=0),
                      'cumle_tam_dogru': d.groupby('id').dogru.min().mean()})

GRUP_SIRA = ['TÜMÜ', 'tek_hedefli', 'ayni_duygulu', 'notrle_karisik', 'zit_duygulu']
def rapor_olustur(t):
    r = pd.DataFrame({g: skor(t[t.grup == g]) for g in GRUP_SIRA[1:]}).T
    r.loc['TÜMÜ'] = skor(t)
    return r.loc[GRUP_SIRA]

def uzunluk_eslenmis(t):
    zit, ayni = t[t.grup == 'zit_duygulu'], t[t.grup == 'ayni_duygulu']
    kesim = np.unique(np.percentile(zit.kelime, [0, 20, 40, 60, 80, 100])); kesim[0], kesim[-1] = 0, np.inf
    agirlik = pd.cut(zit.kelime, kesim).value_counts(normalize=True)
    ayni_acc = ayni.groupby(pd.cut(ayni.kelime, kesim), observed=False).dogru.mean()
    return pd.Series({'zıt duygulu': zit.dogru.mean(), 'aynı duygulu (ham)': ayni.dogru.mean(),
                      'aynı duygulu (uzunluk eşlenmiş)': (ayni_acc * agirlik).sum()})

def kopya_orani(t):
    zit = t[t.grup == 'zit_duygulu']
    etiketler = zit.groupby('id').apply(lambda g: list(zip(g.sira, g.label)))
    hata = zit[zit.dogru == 0]
    kopya = [int(r.pred in {l for s, l in etiketler[r.id] if s != r.sira}) for r in hata.itertuples()]
    return pd.Series({'zıt hatası': len(hata), 'diğer target kopyası (%)': 100 * np.mean(kopya) if kopya else np.nan})

meta = pairs[['id', 'sira', 'grup', 'kelime', 'baglac']]
sonuclar = {}
for kisa in MODELLER:
    f = f'{SONUC_DIR}/{kisa}_test_tahminleri.csv'
    if os.path.exists(f):
        t = pd.read_csv(f)[['id', 'sira', 'label', 'pred']].merge(meta, on=['id', 'sira'])
        t['dogru'] = (t.label == t.pred).astype(int)
        sonuclar[kisa] = t
print('Karşılaştırılan modeller:', list(sonuclar))

for metrik in ['accuracy', 'macro_f1', 'cumle_tam_dogru']:
    tablo = pd.DataFrame({k: rapor_olustur(t)[metrik] for k, t in sonuclar.items()}).round(4)
    print(f'\n=== {metrik} ==='); display(tablo)
    tablo.to_csv(f'{SONUC_DIR}/karsilastirma_{metrik}.csv')

fark = pd.DataFrame({k: {'aynı duygulu': rapor_olustur(t).loc['ayni_duygulu', 'cumle_tam_dogru'],
                         'zıt duygulu':  rapor_olustur(t).loc['zit_duygulu',  'cumle_tam_dogru']} for k, t in sonuclar.items()}).T
fark['fark (puan)'] = 100 * (fark['aynı duygulu'] - fark['zıt duygulu'])
print('\n=== Cümle tam doğruluk: aynı vs zıt duygulu ==='); display(fark.round(4))

uz = pd.DataFrame({k: uzunluk_eslenmis(t) for k, t in sonuclar.items()}).round(4)
print('\n=== Uzunluk eşlenmiş accuracy ==='); display(uz)

bg = pd.DataFrame({k: t[t.grup == 'zit_duygulu'].groupby('baglac').dogru.mean() for k, t in sonuclar.items()})
bg.index = ['bağlaçsız zıt cümle', 'bağlaçlı zıt cümle']
print('\n=== Zıt duygulu: bağlaç kırılımı (accuracy) ==='); display(bg.round(4))

kp = pd.DataFrame({k: kopya_orani(t) for k, t in sonuclar.items()}).round(1)
print('\n=== Zıt duygulu hatalar: diğer target\'ın duygusunu kopyalama ==='); display(kp)

for ad, tb in [('fark', fark), ('uzunluk', uz), ('baglac', bg), ('kopya', kp)]:
    tb.to_csv(f'{SONUC_DIR}/karsilastirma_{ad}.csv')
print('Kaydedildi:', SONUC_DIR)

Karşılaştırılan modeller: ['berturk', 'modernbert_tr', 'xlmr']

=== accuracy ===


,berturk,modernbert_tr,xlmr
TÜMÜ,0.8781,0.8772,0.8681
tek_hedefli,0.9245,0.9119,0.8553
ayni_duygulu,0.9344,0.9297,0.9205
notrle_karisik,0.7542,0.7550,0.7542
zit_duygulu,0.8696,0.8772,0.8662



=== macro_f1 ===


,berturk,modernbert_tr,xlmr
TÜMÜ,0.8598,0.8594,0.8512
tek_hedefli,0.9195,0.9072,0.8476
ayni_duygulu,0.9197,0.9150,0.9050
notrle_karisik,0.7577,0.7587,0.7581
zit_duygulu,0.7128,0.7356,0.7301



=== cumle_tam_dogru ===


,berturk,modernbert_tr,xlmr
TÜMÜ,0.7601,0.7646,0.7436
tek_hedefli,0.9245,0.9119,0.8553
ayni_duygulu,0.8734,0.8674,0.8475
notrle_karisik,0.4832,0.5056,0.4989
zit_duygulu,0.7194,0.7372,0.7117


... (6 outputs left out)


=== Zıt duygulu hatalar: diğer target'ın duygusunu kopyalama ===


/tmp/ipykernel_554/918166632.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  etiketler = zit.groupby('id').apply(lambda g: list(zip(g.sira, g.label)))
/tmp/ipykernel_554/918166632.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  etiketler = zit.groupby('id').apply(lambda g: list(zip(g.sira, g.label)))
/tmp/ipykernel_554/918166632.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the gro

,berturk,modernbert_tr,xlmr
zıt hatası,153.0,144.0,157.0
diğer target kopyası (%),72.5,74.3,68.2


Kaydedildi: /content/drive/MyDrive/cse654_absa/sonuclar
